In [1]:
import pandas as pd
df=pd.read_csv("/content/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
#tokenize
def tokenize(text):
  text=text.lower()
  text=text.replace("?","")
  text=text.replace(" ' ","")
  return text.split()

In [4]:
vocab={'<UNK>':0}

In [5]:
#vocabualry
def build_vocab(row):
  tokenized_question=tokenize(row['question'])
  tokenized_answer=tokenize(row['answer'])
  mrged_tokens=tokenized_question+tokenized_answer
  for token in mrged_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)


In [7]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [8]:
len(vocab)

326

In [26]:
def text_to_indices(text,vocab):
  indexed_token=[]
  for token in tokenize(text):
    if token in vocab:
      indexed_token.append(vocab[token])
    else:
      indexed_token.append(vocab['<UNK>'])

  return indexed_token

In [27]:
import torch
from torch.utils.data import Dataset,DataLoader

In [32]:
class QaDataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab
  def __len__(self):
    return df.shape[0]
  def __getitem__(self,index):
    numerical_question=text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer=text_to_indices(self.df.iloc[index]['answer'],self.vocab)
    return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [33]:

dataset=QaDataset(df,vocab)

In [34]:
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [35]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [36]:
for question,answer in dataloader:
  print (question,answer)

tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([[134]])
tensor([[ 1,  2,  3, 37, 38, 39, 40]]) tensor([[41]])
tensor([[ 42,  18,   2,   3, 283, 143,   3, 284]]) tensor([[206]])
tensor([[ 42, 137,   2, 138,  39, 176, 271]]) tensor([[99]])
tensor([[10, 75, 76]]) tensor([[77]])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]) tensor([[28]])
tensor([[ 42, 252, 253, 118, 254, 255]]) tensor([[256]])
tensor([[  1,   2,   3,   4,   5, 207]]) tensor([[208]])
tensor([[  1,   2,   3, 181, 182, 183, 184]]) tensor([[185]])
tensor([[  1,   2,   3,   4,   5, 238, 239]]) tensor([[240]])
tensor([[  1,   2,   3, 235,   5, 236]]) tensor([[237]])
tensor([[ 42, 137,   2,  62,  39,   3, 324, 325]]) tensor([[6]])
tensor([[ 10,  75, 209]]) tensor([[210]])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([[85]])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([[100]])
tensor([[ 10,  29, 130, 131]]) tensor([[132]])
tensor([[42, 18,  2, 62, 63,  3, 64, 18]]) tensor([[65]])
tensor([[ 10,  11, 158, 159, 160]]) tensor([

In [50]:
import torch.nn as nn
class MyRnn(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)
    self.Linear=nn.Linear(64,vocab_size)

  def forward(self,question):
    embedded_question=self.embedding(question)
    hidden,final=self.rnn(embedded_question)
    output=self.Linear(final.squeeze(0))

    return output



In [51]:
learning_rate=0.001
epoch=20

In [52]:
model=MyRnn(len(vocab))


In [53]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [55]:
for epoch in range(epochs):

  total_loss=0

  for question,answer in dataloader:
    optimizer.zero_grad()
    output=model(question)
    loss=criterion(output,answer[0])
    loss.backward()
    optimizer.step()
    total_loss=total_loss+loss.item()
  print(f"epochs:{epoch+1},loss:{total_loss:.4f}")

epochs:1,loss:528.1167
epochs:2,loss:458.1766
epochs:3,loss:380.0635
epochs:4,loss:319.5615
epochs:5,loss:269.2336
epochs:6,loss:222.8066
epochs:7,loss:178.8300
epochs:8,loss:139.8903
epochs:9,loss:107.7769
epochs:10,loss:82.3571
epochs:11,loss:63.0567
epochs:12,loss:49.2527
epochs:13,loss:38.6755
epochs:14,loss:31.1109
epochs:15,loss:25.2605
epochs:16,loss:20.9982
epochs:17,loss:17.5376
epochs:18,loss:14.8457
epochs:19,loss:12.7299
epochs:20,loss:11.0080


In [76]:
def predict(model, question, threshold=0.5):

    numerical_question = text_to_indices(question, vocab)
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)

    output = model(question_tensor)

    probabilities = torch.softmax(output, dim=-1)

    value, index = torch.max(probabilities, dim=-1)

    value = value.item()
    index = index.item()

    print("Predicted index:", index)
    print("Predicted word:", list(vocab.keys())[index])
    print("Probability:", value)

In [81]:
predict(model,"what is the capital of france")

Predicted index: 7
Predicted word: paris
Probability: 0.8923613429069519


In [83]:
print(list(vocab.keys())[7])

paris
